## Stage 0 — Setup

Imports and header so worth athletics doesn't pick it up as a bot.

In [ ]:
import requests
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
from io import StringIO

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

## Stage 1 — Scrape a single race
Here we scrap the races from the results page and get this information:
athlete, country, time, record flag 
- reaction time
- wind reading
- date, venue, meet name

Handles non-numeric results (DNF/DQ/DNS) by keeping the row with a NaN time,
since a DNF is still a real data point (something went wrong that race).

In [ ]:
def clean_mark(mark):
    """Split '9.79 PB' into (9.79, 'PB'), or '9.91' into (9.91, None).
    Non-numeric marks (DNF/DQ/DNS) get NaN time and the code as the flag."""
    if pd.isna(mark):
        return np.nan, None
    parts = str(mark).strip().split() # whitespace
    try:
        time = float(parts[0]) 
        flag = parts[1] if len(parts) > 1 else None
    except ValueError:
        time = np.nan
        flag = parts[0]  # e.g. 'DNF', 'DQ', 'DNS'
    return time, flag


def scrape_race(url):
    resp = requests.get(url, headers=HEADERS, timeout=15) # timeout in seconds
    resp.raise_for_status() # raises an error if the request failed

    tables = pd.read_html(StringIO(resp.text)) # parse all tables from the HTML
    results_table = max(tables, key=len) # get largest tble

    results_table[["time", "record_flag"]] = results_table["MARK"].apply(
        lambda m: pd.Series(clean_mark(m))
    )

    soup = BeautifulSoup(resp.text, "html.parser") # parse the HTML with BeautifulSoup

    # wind
    heading = soup.find(string=lambda s: s and "Wind" in s) # find wind 
    wind = None
    if heading:
        match = re.search(r"Wind\s*([+-]?\d+\.\d+)", heading) # regex for finding wind speed
        if match:
            wind = float(match.group(1))

    # date — from EventDate meta tag 
    date = None
    for meta in soup.find_all("meta"):
        if meta.get("name") == "EventDate" or meta.get("property") == "EventDate":
            date = meta.get("content")
            break

    # venue + meet name
    desc_tag = soup.find("meta", attrs={"property": "og:description"})
    venue = None
    if desc_tag and desc_tag.get("content"):
        venue_match = re.search(r"in\s+([A-Za-z\s]+)$", desc_tag["content"].strip())
        if venue_match:
            venue = venue_match.group(1).strip()

    title_tag = soup.find("meta", attrs={"property": "og:title"})
    meet_name = title_tag["content"].strip() if title_tag and title_tag.get("content") else None

    results_table["wind"] = wind
    results_table["date"] = date
    results_table["venue"] = venue
    results_table["meet_name"] = meet_name
    results_table["source_url"] = url

    return results_table[["ATHLETE", "COUNTRY", "time", "record_flag", "Reaction Time",
                            "wind", "date", "venue", "meet_name", "source_url"]]

## Stage 2 — Test across multiple races

One race working isn't enough proof — different meets can have messier data
(DNF/DQ results) or slightly different page structures. Here we run the
scraper across 4 different races to find where it breaks.

In [ ]:
urls = [
    "https://worldathletics.org/results/olympic-games/2024/the-xxxiii-olympic-games-7153115/men/100-metres/final/result",  # Paris 2024
    "https://worldathletics.org/results/world-athletics-championships/2023/world-athletics-championships-budapest-2023-7138987/men/100-metres/final/result",  # Budapest 2023
    "https://worldathletics.org/results/olympic-games/2021/the-xxxii-olympic-games-7132391/men/100-metres/final/result",  # Tokyo 2020(21) - has a DNF
    "https://worldathletics.org/competitions/world-athletics-championships/world-athletics-championships-oregon-2022-7137279/results/men/100-metres/final/result",  # Oregon 2022 - different URL structure
]

all_races = []
for u in urls:
    try:
        race = scrape_race(u)
        all_races.append(race)
        print(f"OK ({len(race)} rows): {u}")
    except Exception as e:
        print(f"FAILED: {u}")
        print(f"   error: {e}")

full_dataset = pd.concat(all_races, ignore_index=True)
full_dataset